# 🛡️ Drishti Kavach: Training Unified 'RailDrishti' Model on Google Colab (GPU)

This notebook trains the **RailDrishti** (YOLO11-seg) model at full **1024×1024 High-Resolution** on a free NVIDIA Tesla T4 GPU.

In [ ]:
# 1. Verify NVIDIA GPU Acceleration
!nvidia-smi

In [ ]:
# 2. Install Ultralytics and dependencies
!pip install ultralytics -q

In [ ]:
# 3. Mount Google Drive and Extract Dataset
from google.colab import drive
import os, zipfile

drive_zip_path = '/content/drive/MyDrive/raildrishti_colab.zip'
local_zip_path = '/content/raildrishti_colab.zip'

if os.path.exists(local_zip_path):
    print("[+] Found zip in session storage! Extracting...")
    with zipfile.ZipFile(local_zip_path, 'r') as zip_ref:
        zip_ref.extractall('/content/')
    print("[+] Dataset unzipped successfully!")
else:
    try:
        drive.mount('/content/drive')
        if os.path.exists(drive_zip_path):
            print("[+] Found zip in Google Drive! Extracting...")
            with zipfile.ZipFile(drive_zip_path, 'r') as zip_ref:
                zip_ref.extractall('/content/')
            print("[+] Dataset unzipped successfully from Google Drive!")
        else:
            print(f"[!] Please upload 'raildrishti_colab.zip' to your Google Drive root folder (MyDrive)!")
    except Exception as e:
        print("[!] Error mounting drive:", e)

In [ ]:
# 4. Generate Colab Dataset Configuration
import yaml

data_yaml = {
    'path': '/content/dataset_rail-drishti',
    'train': 'images/train',
    'val': 'images/val',
    'names': {
        0: 'Rail_Track_Bed',
        1: 'Rail_Lines',
        2: 'Branch',
        3: 'IronRod',
        4: 'Barrel',
        5: 'Boulder',
        6: 'Jerrycan',
        7: 'Person',
        8: 'Cattle',
        9: 'Animal',
        10: 'Vehicle'
    }
}

with open('/content/raildrishti_colab.yaml', 'w') as f:
    yaml.dump(data_yaml, f, sort_keys=False)

print("Created /content/raildrishti_colab.yaml")

In [ ]:
# 5. Train Unified 'RailDrishti' Model (YOLO11-seg, Full 1024 Res, 40 Epochs, Zero RAM Crash)
from ultralytics import YOLO

model = YOLO('yolo11s-seg.pt')

results = model.train(
    data='/content/raildrishti_colab.yaml',
    epochs=40,
    imgsz=1024,
    batch=8,
    device=0,
    name='RailDrishti_Training',
    workers=2,
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    cache=False,  # Reads from Colab's fast NVMe SSD (Prevents Colab RAM crash)
    amp=True,
    save=True,
    patience=12,
    verbose=True
)

In [ ]:
# 6. Save and Download Best Trained Model
from google.colab import files
import os, shutil

best_weight = '/content/runs/segment/RailDrishti_Training/weights/best.pt'
if os.path.exists(best_weight):
    shutil.copy(best_weight, '/content/RailDrishti.pt')
    if os.path.exists('/content/drive/MyDrive'):
        shutil.copy(best_weight, '/content/drive/MyDrive/RailDrishti.pt')
        print("[+] Permanent backup saved to your Google Drive: MyDrive/RailDrishti.pt")
    print("Downloading RailDrishti.pt to your computer...")
    files.download('/content/RailDrishti.pt')
else:
    print("Training not finished or weight not found.")